# Phase 5 — LangGraph workflow (retrieve → grade → generate → verify → not-found)

Wires Phase 2 (retrieval), Phase 3 (`get_llm`) and Phase 4 (prompt v3) into
one `graph.invoke({"question": ...})` call, with two gates: `grade` before
generation (are the chunks relevant at all?) and `verify` after (does the
answer actually cite a page?). See
`learning/07_langgraph/01_langgraph_workflow.md` for the why.

## Build the collection

Uses `src.retrieval.RetrievalPipeline` (new in this task) instead of
duplicating ingest code inline, since the graph needs the same pipeline
object at runtime, not just at notebook-setup time.

In [1]:
import logging
import os
import sys
from pathlib import Path

logging.getLogger("numexpr").setLevel(logging.WARNING)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

sys.path.insert(0, str(Path.cwd().parent))

from generate_sample_pdf import generate

from src.retrieval import RetrievalPipeline

pdf_path = generate()
pipeline = RetrievalPipeline(storage_path=Path("data/qdrant"))
chunk_count = pipeline.ingest(pdf_path)
print(f"Ingested {chunk_count} chunks")

/Users/rtlecl910010/Desktop/Projects/Project/papermind/backend/.venv/lib/python3.11/site-packages/huggingface_hub/utils/tqdm.py:176: UserWarning: Cannot enable progress bars: environment variable `HF_HUB_DISABLE_PROGRESS_BARS=1` is set and has priority.
  warnings.warn(


Ingested 4 chunks


## Build the graph

Needs `backend/.env` with `OPENAI_API_KEY`.

In [2]:
from src.graph import build_graph
from src.llm import get_llm

llm = get_llm("openai")
graph = build_graph(pipeline, llm)

## Case 1 — grounded question

Same question used throughout Phase 4. Expect: `grade` says relevant,
`generate` produces a cited answer, `verify` confirms a citation is
present, final state reaches END with that answer.

In [3]:
grounded_result = graph.invoke({"question": "How is citation accuracy measured in this paper?"})
print("chunks:", [c["page_label"] for c in grounded_result["chunks"]])
print("grounded:", grounded_result["grounded"])
print("answer:", grounded_result["answer"])

2026-11-25 19:19:08,292 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-11-25 19:19:09,885 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


chunks: ['1', '4']
grounded: True
answer: The context provided does not include specific information on how citation accuracy is measured in the paper. It only mentions that combining dense retrieval with a re-ranking stage improves citation accuracy compared to retrieval alone (page 1) and that grounded answers with page citations are rated as more trustworthy than ungrounded answers by human reviewers (page 4). However, there are no details on the actual measurement methodology for citation accuracy.


## Case 2 — question completely outside the document's scope

Expect: `grade` says not relevant (the retrieved chunks are the closest
match hybrid search can find, but have nothing to do with this question),
routes straight to `not_found`, no `generate`/`verify` call happens at
all.

In [4]:
offtopic_result = graph.invoke({"question": "What is the capital city of France?"})
print("chunks:", [c["page_label"] for c in offtopic_result["chunks"]])
print("grounded:", offtopic_result["grounded"])
print("answer:", offtopic_result["answer"])

2026-11-25 19:19:10,740 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


chunks: ['3', '1']
grounded: False
answer: I don't have enough information in the uploaded document to answer that.


## Case 3 — topically close, but the specific fact isn't in the document

This is the harder case flagged in task #13: not totally unrelated (so
`grade` may well let it through to `generate`), but the document never
states a funding source. Expect either `grade` catches it, or `generate`
produces an uncited refusal that `verify` then routes to `not_found` —
either path should end at the same safe message.

In [5]:
funding_result = graph.invoke({"question": "What funding source supported this research?"})
print("chunks:", [c["page_label"] for c in funding_result["chunks"]])
print("grounded:", funding_result["grounded"])
print("answer:", funding_result["answer"])

2026-11-25 19:19:11,636 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


chunks: ['4', '1']
grounded: False
answer: I don't have enough information in the uploaded document to answer that.


## Manual check

Honest read of the three results above — does `grade` actually filter
case 2, does `verify` catch anything `grade` lets through ungrounded in
case 3, and does case 1 still answer normally end to end? Goes into the
learning doc's "What we did".

## Next

Phase 6 wraps this graph in a FastAPI service (plus a separate Flask
ingestion service), so a question can come in over HTTP instead of a
notebook cell.